# B-Stock quickstart

## 1. Install Python deps (once per venv)

In a terminal, from the **repo root** (`ecothrift-dashboard`):

```powershell
.\venv\Scripts\Activate.ps1
pip install -r workspace/notebooks/_shared/requirements-notebooks.txt
```

Optional Playwright browser fallback: `playwright install chromium`

Use the **same venv** as your Jupyter kernel (Kernel → Change kernel → your project Python).

## 2. Create local config (secrets, not committed)

1. In File Explorer, go to `workspace/notebooks/bstock-scraper/Scraper/`.
2. Copy **`config.example.py`** → **`bstock_config_local.py`** (same folder).
3. Edit **`bstock_config_local.py`**:
   - **`TOKEN`**: In Chrome, log into [bstock.com](https://bstock.com), open **https://bstock.com/all-auctions**, **F12 → Network → Fetch/XHR**, reload. Click a request whose **Response** is **JSON** (not HTML). **Copy as cURL** and copy the long JWT from the `Authorization: Bearer ...` header (paste only the token string, not the word `Bearer`).
   - **`API_URL`**: Use the **exact URL** of that same JSON request. Do **not** guess `https://bstock.com/api/...` (often HTML). Real APIs are often on **`search.bstock.com`** or **`auction.bstock.com`**.
   - Uncomment/add **`Referer`** / **`Origin`** in **`EXTRA_HEADERS`** if your cURL includes them.

## 3. Run this notebook

The **next cell** adds `workspace/notebooks/bstock-scraper` to `sys.path` so `import Scraper` works. If it errors, set **`BSTOCK_PROJECT_ROOT`** manually to your absolute path, e.g. `Path(r"E:\ecothrift-dashboard\workspace\notebooks\bstock-scraper")`.

In [1]:
import sys
from pathlib import Path

# If auto-detect fails, set this to your absolute path to workspace/notebooks/bstock-scraper:
BSTOCK_PROJECT_ROOT = None  # e.g. Path(r"E:\ecothrift-dashboard\workspace\notebooks\bstock-scraper")


def resolve_bstock_project_root() -> Path:
    if BSTOCK_PROJECT_ROOT is not None:
        p = Path(BSTOCK_PROJECT_ROOT).resolve()
        if not (p / "Scraper" / "config.example.py").is_file():
            raise FileNotFoundError(f"Scraper package not found under {p}")
        return p
    cwd = Path.cwd().resolve()
    for d in [cwd, *cwd.parents]:
        marker = d / "bstock-scraper" / "Scraper" / "config.example.py"
        if marker.is_file():
            return d / "bstock-scraper"
        legacy = d / "Scraper" / "config.example.py"
        if legacy.is_file():
            return d
    repo_guess = cwd / "workspace" / "notebooks" / "bstock-scraper"
    if (repo_guess / "Scraper" / "config.example.py").is_file():
        return repo_guess.resolve()
    raise FileNotFoundError(
        "Could not find bstock-scraper (folder containing Scraper/). "
        "Set BSTOCK_PROJECT_ROOT in this cell to the absolute path to workspace/notebooks/bstock-scraper."
    )


bstock_root = resolve_bstock_project_root()
if str(bstock_root) not in sys.path:
    sys.path.insert(0, str(bstock_root))

from Scraper import BStockScraper

scraper = BStockScraper()
auctions = scraper.get_auctions()
auctions.head()

Fetching page 1... got 100 rows (total: 100)
Fetching page 2... got 100 rows (total: 200)
Fetching page 3... got 100 rows (total: 300)
Fetching page 4... got 100 rows (total: 400)
Fetching page 5... got 100 rows (total: 500)
Fetching page 6... got 91 rows (total: 591)
Fetching page 7... [WARNING] No auction list found. Top-level keys: ['listings', 'total', 'offset', 'limit']
Response preview:
{
  "listings": [],
  "total": 591,
  "offset": 600,
  "limit": 100
}
empty page - done.


,accessories,allowedShipmentTypes,auctionId,auctionType,auctionUrl,bfileEnabled,bidBasis,bidUnits,bstockGrade,businessUnit,...,storefrontId,storefrontName,termsRestrictions,title,transportMode,typicalSellingPrice,units,winningBidAmount,id,sellerCountry
0,[],[BINDING],69bb2e3a85f6e4eae265bbf9,PROXY,https://bstock.com/buy/listings/details/69bb2e...,F,PRICE,None,[],Revolve 3rd party,...,69795432a5750ddf3df36eb8,Contemporary Fashion,[],"1 Pallet of Dresses, Outerwear, Tops & More by...",LTL,16465.0,452.0,7925.0,69bb2e3a797f9010f41cb96e,US
1,[],[BUYER_ARRANGED],None,N,https://bstock.com/mobilecarrier/auction/aucti...,F,PRICE,None,[],None,...,att,Mobile Carrier,[],Apple Cell Phones - iPhone X & iPhone 11 Pro -...,LTL,NaN,5.0,376.0,att251166,US
2,[],[BUYER_ARRANGED],None,N,https://bstock.com/mobilecarrier/auction/aucti...,F,PRICE,None,[],None,...,att,Mobile Carrier,[],"Google Cell Phones - Pixel 8a, Pixel 9, Pixel ...",LTL,NaN,44.0,11733.0,att251098,US
3,[],[BUYER_ARRANGED],None,N,https://bstock.com/mobilecarrier/auction/aucti...,F,PRICE,None,[],None,...,att,Mobile Carrier,[],Google Cell Phones - Pixel 9 Pro XL - 46 Units...,LTL,NaN,46.0,19333.0,att250735,US
4,[],[BUYER_ARRANGED],None,N,https://bstock.com/mobilecarrier/auction/aucti...,F,PRICE,None,[],None,...,att,Mobile Carrier,[],Emblem Cell Phones - Motivate Pro 2 & Verge 2 ...,LTL,NaN,9.0,527.0,att250652,US


In [ ]:
fresh = scraper.update()
fresh.head()

In [ ]:
# scraper.get_manifests(fresh)  # NotImplementedError until manifest API is wired

In [2]:
# Per marketplace: distinct auctions vs rows (rows > distinct ⇒ duplicate pulls / same auction repeated)
if "siteName" in auctions.columns and "auctionId" in auctions.columns:
    by_site = (
        auctions.groupby("siteName", dropna=False)
        .agg(distinct_auctions=("auctionId", "nunique"), rows=("auctionId", "count"))
        .sort_values("distinct_auctions", ascending=False)
    )
    by_site
else:
    print("Missing siteName or auctionId — inspect auctions.columns")

# One-liner: row counts only
# auctions["siteName"].value_counts(dropna=False)

# Exclude sites you do not care about (then re-run summaries on `filtered`)
# IGNORE = {"Mobile Carrier Liquidations"}
# filtered = auctions.loc[~auctions["siteName"].isin(IGNORE)]

In [5]:
auctions.groupby("siteName", dropna=False).count()

,accessories,allowedShipmentTypes,auctionId,auctionType,auctionUrl,bfileEnabled,bidBasis,bidUnits,bstockGrade,businessUnit,...,storefrontId,storefrontName,termsRestrictions,title,transportMode,typicalSellingPrice,units,winningBidAmount,id,sellerCountry
siteName,,,,,,,,,,,,,,,,,,,,,
Amazon UK,38,38,0,38,38,38,38,0,38,0,...,38,38,38,38,38,34,35,38,38,38
Att,14,14,0,14,14,14,14,0,14,0,...,14,14,14,14,14,0,14,14,14,14
Essendant,2,2,0,2,2,2,2,0,2,0,...,2,2,2,2,2,2,2,2,2,2
Gamestop,2,2,0,2,2,2,2,0,2,0,...,2,2,2,2,2,0,2,2,2,2
QVC,19,19,0,19,19,19,19,0,19,0,...,19,19,19,19,19,9,19,19,19,19
TMP,489,489,463,463,489,489,489,9,489,134,...,489,489,489,489,489,334,489,456,489,489
Veyer,2,2,0,2,2,2,2,0,2,0,...,2,2,2,2,2,2,2,2,2,2
bigfive,1,1,0,1,1,1,1,0,1,0,...,1,1,1,1,1,1,1,1,1,1
dickssportinggoods,3,3,0,3,3,3,3,0,3,0,...,3,3,3,3,3,3,3,3,3,3
